# Professional English Coach — media review pilot

Use this notebook in Google Colab to inspect a **pre-authored lesson audio candidate** before considering it for the static site. It does not generate speech, call VoiceStudio, or connect to a learner-facing service. It reports technical metadata and lets a human listen and record review decisions.

**Privacy:** Colab runs notebook code on a Google-managed runtime. Upload only authored lesson clips that are appropriate to process there. Never upload learner voice recordings, private reference voices, credentials, or confidential material. Do not mount Google Drive for this review.

The current candidate is in `content/media-candidates/tradeoff-phrase.mp3`. The site build excludes candidate media. Keep it excluded unless pronunciation, transcript, and redistribution rights are independently verified and recorded.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select one authored lesson-media candidate.
if not uploaded:
    raise RuntimeError("Upload one approved-for-review lesson candidate first.")
filename = next(iter(uploaded))
clip_path = Path(filename)
print(f"Loaded {clip_path.name} ({clip_path.stat().st_size:,} bytes)")


In [ ]:
import json
import mimetypes
import shutil
import subprocess

ffprobe = shutil.which("ffprobe")
if not ffprobe:
    raise RuntimeError("ffprobe is unavailable in this runtime; inspect the file locally and record metadata manually.")
result = subprocess.run(
    [ffprobe, "-v", "error", "-show_entries", "format=duration,size:stream=codec_name,sample_rate,channels", "-of", "json", str(clip_path)],
    check=True, capture_output=True, text=True
)
metadata = json.loads(result.stdout)
print(json.dumps(metadata, indent=2))


In [ ]:
from IPython.display import Audio, display

display(Audio(filename=str(clip_path)))


## Human review checklist

Listen to the **final encoded file** above, not only the source script. Check:

1. The spoken sentence matches the approved script word for word.
2. Technical terms, connected speech, stress, pace and word boundaries are understandable at normal playback speed.
3. The written transcript is accurate and can be revealed after submission; the learner can replay the clip and can skip the optional activity.
4. The generation tool and exact model/voice terms allow the intended redistribution. For a cloned voice, confirm explicit permission from the voice owner.
5. The clip is short, plays in target desktop/mobile browsers, and has an accessible text route.

A notebook checkbox or generated manifest is a record of a reviewer’s decision, not a substitute for checking the applicable licenses. If anything is uncertain, leave the status pending and do not publish.


In [ ]:
# Fill these only after you have performed each check above.
reviewer = ""  # Name or team identifier
reviewed_on = ""  # YYYY-MM-DD
script = "The main trade-off is higher latency in exchange for better isolation."
script_matches = False
pronunciation_is_clear = False
transcript_matches_audio = False
redistribution_terms_verified = False
voice_owner_permission_confirmed = True  # True only when no cloned/reference voice is used or permission is recorded
source_record = "FFmpeg 6.1.1 / libflite voice kal; local synthetic voice; no reference recording"
model_or_voice_license = ""  # Exact license/terms source checked by reviewer


In [ ]:
from datetime import date

checks_pass = all([
    bool(reviewer.strip()),
    bool(reviewed_on.strip()),
    script_matches,
    pronunciation_is_clear,
    transcript_matches_audio,
    redistribution_terms_verified,
    voice_owner_permission_confirmed,
    bool(model_or_voice_license.strip()),
])
manifest = {
    "candidateFile": clip_path.name,
    "contentType": mimetypes.guess_type(clip_path.name)[0],
    "script": script,
    "sourceRecord": source_record,
    "modelOrVoiceLicense": model_or_voice_license or None,
    "humanReview": {
        "reviewer": reviewer or None,
        "reviewedOn": reviewed_on or None,
        "scriptMatchesAudio": script_matches,
        "pronunciationIsClear": pronunciation_is_clear,
        "transcriptMatchesAudio": transcript_matches_audio,
        "redistributionTermsVerified": redistribution_terms_verified,
        "voiceOwnerPermissionConfirmed": voice_owner_permission_confirmed,
        "status": "approved" if checks_pass else "pending",
    },
    "publicationStatus": "eligible-for-project-review" if checks_pass else "excluded-from-static-build",
}
output_path = Path("media-review-result.json")
output_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(output_path.read_text())
print("This result is not deployed. Copy it into the project only after checking the evidence and updating the reviewed asset manifest.")


In [ ]:
files.download(str(output_path))
